In [1]:
! pip install ultralytics insightface onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 28.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s 

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from insightface.app import FaceAnalysis
import os
import pandas as pd
import csv

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kibordevents/iranian-national-id-card-dataset")

In [6]:
detector = FaceAnalysis(name='buffalo_l') # set face detector
detector.prepare(ctx_id=0, det_size=(640, 640))
card_model = YOLO('/content/drive/MyDrive/NationalCard-imageProject/models/CardDetection.pt') # set yolo model for card detection
lst = []

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 103199.96KB/s]
/usr/local/lib/python3.11/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:121: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [7]:
csv_file = "/content/drive/MyDrive/NationalCard-imageProject/angles.csv"
if not os.path.exists(csv_file):
    with open(csv_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "angle"])

In [8]:
def rotation(path):
    image = cv2.imread(path)  # set image
    results = card_model(image)  # run card yolo model for image

    if len(results[0].boxes) == 0:  # if cant recognize card
        print("No card detected!")
        print('--------------------------------------------------------------')
        print(path)
        return None

    # set 4 corner location
    x1, y1, x2, y2 = map(int, results[0].boxes[0].xyxy[0])
    cropped_card = image[y1:y2, x1:x2]

    # detect face in the cropped card
    faces = detector.get(cropped_card)

    # check for faces
    if faces:
        x1, y1, x2, y2 = map(int, faces[0]['bbox'])  # get face bounding box
        w, h = x2 - x1, y2 - y1
        face_center = (x1 + w // 2, y1 + h // 2)  # face center
        card_center = (cropped_card.shape[1] // 2, cropped_card.shape[0] // 2)  # card center

        # calculate rotation angle
        angle = np.arctan2(face_center[1] - card_center[1], face_center[0] - card_center[0]) * (180 / np.pi) + 180
        rotation_matrix = cv2.getRotationMatrix2D(card_center, angle, 1.0)
        rotated = cv2.warpAffine(cropped_card, rotation_matrix, (cropped_card.shape[1], cropped_card.shape[0]))


        # Write to CSV
        with open(csv_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([os.path.basename(path), round(angle, 2)])

        print(path)
        print('--------------------------------------------------------------')
        print(angle)
        return rotated

    print('cant find face')
    print('--------------------------------------------------------------')
    lst.append(path)
    print(path)
    return cropped_card


In [9]:
def run_on_folder(folder_path, extensions=[".jpg", ".jpeg", ".png"]):
    for filename in os.listdir(folder_path):
        if any(filename.lower().endswith(ext) for ext in extensions):
            file_path = os.path.join(folder_path, filename)
            rotation(file_path)

In [14]:
run_on_folder('/kaggle/input/iranian-national-id-card-dataset/iranian_national_id_card_images/iranian_national_id_card_images')

Streaming output truncated to the last 5000 lines.
--------------------------------------------------------------
268.83086067209257

0: 480x640 1 Card, 14.2ms
Speed: 6.6ms preprocess, 14.2ms inference, 6.2ms postprocess per image at shape (1, 3, 480, 640)
cant find face
--------------------------------------------------------------
/kaggle/input/iranian-national-id-card-dataset/iranian_national_id_card_images/iranian_national_id_card_images/618.jpg

0: 480x640 1 Card, 13.2ms
Speed: 4.4ms preprocess, 13.2ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)
cant find face
--------------------------------------------------------------
/kaggle/input/iranian-national-id-card-dataset/iranian_national_id_card_images/iranian_national_id_card_images/1051.jpg

0: 480x640 1 Card, 13.3ms
Speed: 4.3ms preprocess, 13.3ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)
/kaggle/input/iranian-national-id-card-dataset/iranian_national_id_card_images/iranian_national_id_c